# 2048 AI - Validacao no Google Colab

Este notebook valida se o Google Colab consegue clonar o projeto, instalar o pacote, importar o motor do jogo e avaliar os agentes de referencia.

Nesta etapa ainda nao vamos treinar uma rede neural. O objetivo e confirmar que a base tecnica esta pronta para o treinamento futuro.

## 1. Preparar ambiente

Execute esta celula primeiro. Ela clona ou atualiza o repositorio, instala o projeto e garante que o kernel atual encontre o pacote `game2048`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Sankofa-JBC/2048-ai.git"
PROJECT_DIR = Path("/content/2048-ai")
SRC_DIR = PROJECT_DIR / "src"

if PROJECT_DIR.exists():
    print("Repositorio ja existe. Atualizando...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)
else:
    print("Clonando repositorio...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

if not (PROJECT_DIR / "pyproject.toml").exists():
    raise FileNotFoundError("pyproject.toml nao encontrado na raiz do projeto clonado.")

print("Instalando projeto em modo editavel...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    cwd=PROJECT_DIR,
    check=True,
)

# Em notebooks, o kernel atual pode nao enxergar imediatamente o caminho
# criado pelo pip editavel. Inserimos src explicitamente para evitar isso.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Raiz do projeto:", PROJECT_DIR)
print("Pacotes Python em:", SRC_DIR)

## 2. Validar importacao do pacote

Se esta celula rodar sem erro, o Colab ja consegue usar o motor do jogo e os agentes.

In [ ]:
from game2048 import Game2048, HeuristicAgent, RandomAgent

game = Game2048(seed=42)
print("OK:", Game2048.__name__, RandomAgent.name, HeuristicAgent.name)
print("Tabuleiro inicial:")
for row in game.board:
    print(row)

## 3. Rodar os testes

Os testes confirmam que as regras do jogo, os agentes e o avaliador continuam funcionando dentro do Colab.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Avaliar os agentes de referencia

Aqui comparamos o agente aleatorio com o agente heuristico usando a mesma seed base.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "game2048.cli.evaluate_agents", "--agent", "random", "--games", "100", "--seed", "42"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "game2048.cli.evaluate_agents", "--agent", "heuristic", "--games", "100", "--seed", "42"],
    check=True,
)

## 5. Exportar metricas

Os arquivos gerados em `results/` serao uteis para comparar experimentos depois.

In [ ]:
(PROJECT_DIR / "results").mkdir(exist_ok=True)

subprocess.run([
    sys.executable,
    "-m",
    "game2048.cli.evaluate_agents",
    "--agent",
    "random",
    "--games",
    "100",
    "--seed",
    "42",
    "--output",
    str(PROJECT_DIR / "results/random_colab.json"),
], check=True)

subprocess.run([
    sys.executable,
    "-m",
    "game2048.cli.evaluate_agents",
    "--agent",
    "heuristic",
    "--games",
    "100",
    "--seed",
    "42",
    "--output",
    str(PROJECT_DIR / "results/heuristic_colab.json"),
], check=True)

## 6. Ler os resultados exportados

Esta celula carrega os JSONs e mostra um resumo simples dentro do notebook.

In [ ]:
import json

for file_name in ["results/random_colab.json", "results/heuristic_colab.json"]:
    with open(PROJECT_DIR / file_name, "r", encoding="utf-8") as file:
        data = json.load(file)

    summary = data["summary"]
    print(file_name)
    print("  agente:", summary["agent_name"])
    print("  jogos:", summary["games"])
    print("  score medio:", round(summary["average_score"], 2))
    print("  melhor score:", summary["best_score"])
    print("  maior bloco:", summary["best_max_tile"])
    print()

## Proximo passo

Se todas as celulas rodarem sem erro, o ambiente Colab esta validado. Depois disso podemos criar o primeiro algoritmo de aprendizagem.